# Capstone — mirrors your deployed research paper

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/capstone.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## Setup

**Method:** this cell finds the project folder (on Colab, it first clones the repository given in `REPO_URL`) and defines four small helpers: one to open a weekly notebook, one to pull a saved table by its title, one to pull a saved chart, and one to check that the numbers quoted in this notebook appear in the tables they come from. It reads no warehouse data and needs no token.


In [ ]:
import base64
import json
import os
import subprocess
import sys

from IPython.display import Markdown, display

REPO_URL = "https://github.com/neilivanorencia/flyrank-ml-internship"  # the GitHub repository holding this project; used only when the notebooks are not found locally (for example on Colab)

# Locally, walk up to the project's top folder (the one holding work/notebooks); on Colab, clone the repository first.
while not os.path.isdir("work/notebooks") and os.path.dirname(os.getcwd()) != os.getcwd():
    os.chdir("..")
if not os.path.isdir("work/notebooks") and "google.colab" in sys.modules and REPO_URL:
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, "/content/project"], check=True)
    os.chdir("/content/project")
assert os.path.isdir("work/notebooks"), "The weekly notebooks were not found; run this from the project folder or set REPO_URL."
NOTEBOOKS = "work/notebooks"
OUTPUTS = "work/outputs"


def notebook(name: str) -> dict:
    """Return a weekly notebook as a dictionary."""
    with open(f"{NOTEBOOKS}/{name}.ipynb", encoding="utf-8") as handle:
        return json.load(handle)


def saved_table(name: str, title: str, rows: int | None = None) -> str:
    """Return the saved Markdown table whose bold title starts with `title`, optionally cut to its first `rows` rows."""
    for cell in notebook(name)["cells"]:
        for output in cell.get("outputs", []):
            text = "".join(output.get("data", {}).get("text/markdown", []))
            if text.startswith(f"**{title}"):
                if rows is not None:
                    lines = text.splitlines()
                    start = next(i for i, line in enumerate(lines) if line.startswith(("| ---", "| :"))) + 1
                    text = "\n".join(lines[:start + rows])
                return text
    raise KeyError(f"{name}: no saved table titled '{title}'")


def saved_chart(name: str, number: int = 0) -> bytes:
    """Return the `number`-th saved chart of a weekly notebook as PNG bytes."""
    charts = [output["data"]["image/png"] for cell in notebook(name)["cells"] for output in cell.get("outputs", [])
              if "image/png" in output.get("data", {})]
    return base64.b64decode(charts[number])


def check(text: str, *values: str) -> None:
    """Fail loudly if a number quoted in this notebook is missing from the table it is quoted from."""
    missing = [value for value in values if value not in text]
    assert not missing, f"Quoted numbers not found in the source table: {missing}"


WEEKLY = {"w01_research_question": "ML-01 to ML-02", "w02_ml_task_framing": "ML-03", "w03_data_contract": "ML-04",
          "w03_feature_leakage_check": "ML-05", "w04_signal_audit": "ML-06", "w04_baseline_score": "ML-07",
          "w05_model": "ML-08", "w06_validation_audit": "ML-09", "w07_action_playbook": "ML-10"}
display(Markdown("**The weekly notebooks this capstone draws on**\n\n| Notebook | Card | Title | Findings |\n| --- | --- | --- | ---: |\n" + "\n".join(
    f"| `{name}` | {card} | {''.join(notebook(name)['cells'][0]['source']).splitlines()[0].lstrip('# ')} | "
    f"{sum(1 for cell in notebook(name)['cells'] if cell['cell_type'] == 'markdown' and ''.join(cell['source']).startswith('**Finding:**'))} |"
    for name, card in WEEKLY.items())))

## 1. Question

*The research question and the decision it supports.*

## 2. Data

*Which release, which tables, date windows, what you excluded and why. Public-safe.*

## 3. Methodology

*Assumptions, features, label definition, baseline, validation design, leakage checks.*

## 4. Results (vs baseline)

*Model vs baseline on the same split. The honest table.*

## 5. Limitations

*What this work cannot claim.*

## 6. Ranked recommendations

*The action playbook output — the paper's recommendations section.*

## 7. Artifacts the paper embeds

*Generate/collect the charts and tables your deployed page will show.*

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.
- [ ] My deployed paper has **all 9 sections** — including the **Abstract** at the top and **Acknowledgments & data credit** (the https://flyrank.ai link) at the bottom.
- [ ] **ML-12 done in this notebook's closing cells:** 5-minute demo outline + a social-post cut + a 3-sentence employer-facing summary.
